# Water dimer: a moving hydrogen bond
Use **ASE + MACE-OFF23-small** for six atoms in vacuum. Plot the O···O distance and
O–H···O angle as the molecules vibrate and reorient. A small initial stretch gives
visible motion without waiting for a rare event. This is a short trajectory, not a
hydrogen-bond lifetime or an equilibrium liquid-water calculation.

Run in the MACE-enabled kernel from this notebook's directory. The helper source is
visible in `../../../demo_tools`; the MD and analysis are written directly below.

In [ ]:
import os, sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from ase import Atoms, units
from ase.build import molecule
from ase.io import read, write
from ase.optimize import BFGS
from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary

sys.path.insert(0, str(Path.cwd().parent.parent.parent))
from demo_tools.helpers import start_exercise
from demo_tools.view_mols import view
_, OUTPUT = start_exercise()
torch.set_num_threads(int(os.environ.get('SLURM_CPUS_PER_TASK', '1')))

from mace.calculators import MACECalculator
calc = MACECalculator(model_paths='../../models/MACE-OFF23_small.model',
                 device='cpu', default_dtype='float64')


## 1. Prepare and relax the dimer
Atom indices: donor O=0, donor H=1, acceptor O=3. The H-bond angle is 180° for a straight O–H···O arrangement.

In [ ]:
atoms = Atoms('OHHOHH', positions=[[0,0,0], [0.9572,0,0], [-0.24,0.927,0],
                                  [2.85,0,0], [3.44,0.75,0], [3.44,-0.75,0]])
atoms.info.update(total_charge=0, total_spin=0)
atoms.calc = calc
assert BFGS(atoms, logfile=str(OUTPUT/'relax.log')).run(fmax=0.02, steps=200)
view(atoms)


## 2. Gently stretch and release
Displace the second water by 0.15 Å along O···O. Run NVE so the added energy stays in the isolated system, up to integration error. Initial velocities correspond to 50 K; six atoms do not define a macroscopic temperature.

In [ ]:
direction = atoms.positions[3] - atoms.positions[0]
atoms.positions[3:] += 0.15 * direction / np.linalg.norm(direction)
MaxwellBoltzmannDistribution(atoms, temperature_K=50, rng=np.random.default_rng(8))
Stationary(atoms)
DT_FS = 0.5
STEPS = 1000
dyn = VelocityVerlet(atoms, DT_FS*units.fs,
                     trajectory=str(OUTPUT/'dimer.traj'), loginterval=5)
start = time.perf_counter()
dyn.run(STEPS)
print(f'Elapsed: {(time.perf_counter()-start)/60:.1f} min')


## 3. Follow the hydrogen bond
Distance and angle change continuously; neither alone is a universal hydrogen-bond definition. If the dimer dissociates, report it rather than assuming it remains bonded.

In [ ]:
frames = read(OUTPUT/'dimer.traj', index=':')
t = np.arange(len(frames))*5*DT_FS
fig, ax = plt.subplots(3,1,figsize=(7,6),sharex=True)
ax[0].plot(t,[a.get_distance(0,3) for a in frames]); ax[0].set_ylabel('O···O (Å)')
ax[1].plot(t,[a.get_angle(0,1,3) for a in frames]); ax[1].set_ylabel('O–H···O (degrees)')
energy = np.array([a.get_total_energy() for a in frames])
ax[2].plot(t,1000*(energy-energy[0])); ax[2].set(xlabel='Time (fs)',ylabel='Energy change (meV)')
plt.tight_layout(); plt.show()
view(frames[::2], animate=True)


## Try and explain
Repeat with a smaller initial stretch or halve the timestep. Does the dimer remain together?
Does energy drift decrease? Why is this not a measurement of bulk-water behavior?